<a href="https://colab.research.google.com/github/OmarAbazeem/CinemaProjectAssignment/blob/main/Labs/Lab4/Lab4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment: Text Generation with RNNs

In this assignment, you will explore text generation using deep learning models, specifically Recurrent Neural Networks (RNNs). Text generation is a fascinating task where the goal is to train a model that can predict the next word in a sequence, ultimately generating coherent and contextually accurate sentences or paragraphs.

You will work with a corpus of text data provided in the tutorial (from TensorFlow), which contains a large collection of Shakespeare’s works. Your objective is to implement a text generation model using RNNs and to evaluate its performance based on various metrics.

<img src="https://drek4537l1klr.cloudfront.net/teofili2/Figures/03fig09_alt.jpg" alt="Drawing"/>

**The models include:**
- Deep RNN (LSTM)
- Deep RNN (GRU)
- Bidirectional RNN


**Evaluation:**

Evaluate the performance of the text generation model using several metrics:
- Perplexity: A measure of how well the probability distribution predicted by the model aligns with the actual distribution of words in the text.
- Generated Text Quality: Subjectively evaluate the quality of the generated text by considering grammar, coherence, and creativity. This can be done by visually inspecting the generated sequences.
- BLUE Score: Character-level BLEU: BLEU can be applied at the character level by treating each character as an n-gram. This is particularly useful when the task involves generating character sequences (like poetry, code, or fine-grained character-based generation).
For example, the sequence “hello” could be evaluated by comparing 1-grams (e.g., “h”, “e”, “l”, “l”, “o”) or higher-order n-grams (e.g., “he”, “el”, “ll”, “lo”).

**Goals:**

- Compare the performance of GRU and LSTM. Is there a significant difference in the results?
- Compare the performance of the unidirectional and bidirectional RNN models.
Which model produces better results?
- Discuss the impact of bidirectional RNNs on text generation tasks.

# Import TensorFlow and other libraries

In [1]:
import tensorflow as tf
import numpy as np
import nltk
from nltk.translate.bleu_score import sentence_bleu
import os
import time

# Dataset

The Shakespeare dataset used for text generation contains a collection of works

---

by William Shakespeare, primarily in the form of plays and sonnets. The text is used to train language models, offering an ideal example for character-level modeling due to its rich, complex language. The dataset is often preprocessed by tokenizing it into individual characters, enabling models to learn the sequential relationships between characters for generating coherent and contextually appropriate text.

To access the dataset, you can use TensorFlow's get_file method as follow:

You can inspect the dataset on [Kaggle](https://www.kaggle.com/datasets/adarshpathak/shakespeare-text/data).

In [2]:
path_to_file = tf.keras.utils.get_file('shakespeare.txt', 'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt')

# Read, then decode for py2 compat.
text = open(path_to_file, 'rb').read().decode(encoding='utf-8')
# length of text is the number of characters in it
print(f'Length of text: {len(text)} characters')

1115394/1115394 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Length of text: 1115394 characters


In [3]:
# Take a look at the first 250 characters in text
print(text[:250])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.



# Process the text

### Vectorize the text

Before training, you need to convert the strings to a numerical representation.

The `tf.keras.layers.StringLookup` layer can convert each character into a numeric ID. It just needs the text to be split into tokens first.

In [4]:
example_texts = ['abcdefg', 'xyz']

chars = tf.strings.unicode_split(example_texts, input_encoding='UTF-8')
chars

<tf.RaggedTensor [[b'a', b'b', b'c', b'd', b'e', b'f', b'g'], [b'x', b'y', b'z']]>


Now create the [`tf.keras.layers.StringLookup`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/StringLookup) layer:

Since the goal of this assignment is to generate text, it will also be important to invert this representation and recover human-readable strings from it. For this you can use tf.keras.layers.StringLookup(..., invert=True).

Note: Here instead of passing the original vocabulary generated with sorted(set(text)) use the get_vocabulary() method of the tf.keras.layers.StringLookup layer so that the [UNK] tokens is set the same way.

In [6]:
def get_string_lookup_layers(vocab):
    """
    Creates StringLookup layers for encoding characters to IDs and decoding IDs back to characters.

    Args:
        vocab (list): List of unique characters in the dataset.

    Returns:
        ids_from_chars (tf.keras.layers.StringLookup): Converts characters to IDs.
        chars_from_ids (tf.keras.layers.StringLookup): Converts IDs back to characters.
    """
    ids_from_chars = tf.keras.layers.StringLookup(vocabulary=vocab, mask_token=None)
    chars_from_ids = tf.keras.layers.StringLookup(
        vocabulary=ids_from_chars.get_vocabulary(),
        invert=True,
        mask_token=None
    )
    return ids_from_chars, chars_from_ids

In [7]:
def text_from_ids(ids, chars_from_ids):
    """
    Converts a sequence of character IDs into a human-readable string.

    Args:
        ids (tf.Tensor): Tensor of character IDs.
        chars_from_ids (tf.keras.layers.StringLookup): StringLookup layer to decode IDs.

    Returns:
        tf.Tensor: Decoded string.
    """
    return tf.strings.reduce_join(chars_from_ids(ids), axis=-1)

# Create training examples and targets

Next divide the text into example sequences. Each input sequence will contain seq_length characters from the text.

For each input sequence, the corresponding targets contain the same length of text, except shifted one character to the right.

So break the text into chunks of seq_length+1. For example, say seq_length is 4 and our text is "Hello". The input sequence would be "Hell", and the target sequence "ello".

To do this first use the tf.data.Dataset.from_tensor_slices function to convert the text vector into a stream of character indices.

For training you'll need a dataset of (input, label) pairs. Where input and label are sequences. At each time step the input is the current character and the label is the next character.

Here's a function that takes a sequence as input, duplicates, and shifts it to align the input and label for each timestep:

In [8]:
def split_input_target(sequence):
    input_text = sequence[:-1]
    target_text = sequence[1:]
    return input_text, target_text

Convert Text to Numerical Sequences

In [9]:
# Initialize mapping layers
vocab = sorted(set(text))
ids_from_chars, chars_from_ids = get_string_lookup_layers(vocab)



# Convert text to character IDs
all_ids = ids_from_chars(tf.strings.unicode_split(text, 'UTF-8'))

# Create a dataset from the character IDs
ids_dataset = tf.data.Dataset.from_tensor_slices(all_ids)

Create Sequences for Training

In [10]:
seq_length = 100  # Length of each training sequence

# Batch sequences (each sequence is seq_length + 1)
sequences = ids_dataset.batch(seq_length + 1, drop_remainder=True)

# Map dataset to input-target format
dataset = sequences.map(split_input_target)

# Print sample input-output pairs
for input_example, target_example in dataset.take(1):
    print("Input :", tf.strings.reduce_join(chars_from_ids(input_example)).numpy().decode('utf-8'))
    print("Target:", tf.strings.reduce_join(chars_from_ids(target_example)).numpy().decode('utf-8'))

Input : First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You
Target: irst Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You 


# Create RNN Models

Before training our text generation models, we need to set up key parameters and prepare the dataset. These parameters control the size of batches, buffer size for shuffling, embedding dimensions, and the number of units in the recurrent layer.

Customizing Parameters for Optimization

- Increase RNN_UNITS if the model is underfitting (not capturing enough detail).
- Decrease BATCH_SIZE if memory usage is too high (e.g., when using large RNN units).
- Adjust EMBEDDING_DIM to experiment with the quality of learned character representations.

Tip: Start with these values, then fine-tune based on model performance and available computational resources!

In [11]:
BATCH_SIZE = 64
BUFFER_SIZE = 10000
VOCAB_SIZE = len(vocab) + 1
DATASET_SIZE = sum(1 for _ in dataset)

shuffled_dataset = dataset.shuffle(BUFFER_SIZE)

train_dataset = shuffled_dataset.take(int(0.9 * DATASET_SIZE) )
val_dataset = shuffled_dataset.skip(int(0.1 * DATASET_SIZE) )

# Prepare dataset for training
training_dataset = (train_dataset
                 .batch(BATCH_SIZE, drop_remainder=True)
                 .prefetch(tf.data.experimental.AUTOTUNE))
# Prepare dataset for validation
validation_dataset = (val_dataset
               .batch(BATCH_SIZE, drop_remainder=True)
               .prefetch(tf.data.experimental.AUTOTUNE))

In [12]:
EMBEDDING_DIM = 256
RNN_UNITS = 512

# LSTM-Based Model

Long Short-Term Memory (LSTM) is a type of Recurrent Neural Network (RNN) designed to capture long-range dependencies in sequential data. Unlike traditional RNNs, which struggle with vanishing gradients, LSTMs use gates (input, forget, and output gates) to regulate the flow of information, making them highly effective for text generation tasks.

This function defines an LSTM-based neural network for text generation.

**Function Overview:**
* [`tf.keras.layers.Embedding`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Embedding): This is the input layer, consisting of a trainable lookup table that maps the numbers of each character to a vector with `embedding_dim` dimensions.
* [`tf.keras.layers.LSTM`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/LSTM): Our LSTM network, with size `units=rnn_units`.
* [`tf.keras.layers.Dense`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense): The output layer, with `vocab_size` outputs.


<img src="https://raw.githubusercontent.com/aamini/introtodeeplearning/2019/lab1/img/lstm_unrolled-01-01.png" alt="Drawing"/>

In [13]:
def build_lstm_model(vocab_size, embedding_dim, rnn_units, batch_size):
    model = tf.keras.Sequential([
        tf.keras.Input(shape=(None,)),
        tf.keras.layers.Embedding(vocab_size, embedding_dim),
        tf.keras.layers.LSTM(rnn_units, return_sequences=True, stateful=False),
        tf.keras.layers.Dense(vocab_size)
    ])
    return model

In [14]:
def build_stacked_lstm_model(vocab_size, embedding_dim, rnn_units, batch_size, num_layers=2):
    layers = [
        tf.keras.Input(shape=(None,)),
        tf.keras.layers.Embedding(vocab_size, embedding_dim)
    ]

    for _ in range(num_layers):
        layers.append(tf.keras.layers.LSTM(rnn_units, return_sequences=True, stateful=False))

    layers.append(tf.keras.layers.Dense(vocab_size))

    model = tf.keras.Sequential(layers)
    return model

**Customization & Optimization**

These experiments are required to improve model performance. You must conduct the following experiments and document your findings:

- Increase RNN_UNITS : This enhances the model’s ability to recognize deeper patterns but may increase training time.
- Experiment with EMBEDDING_DIM : Adjusting this can improve the quality of character representation.
- Stack multiple LSTM layers : This can help the model understand text more effectively.

**Submission Requirements**

- Perform at least three trials varying the parameters above.
- Keep the most performing plots and outputs in your Jupyter Notebook.


# GRU-Based Model

What is GRU?

Gated Recurrent Units (GRU) are a simplified version of LSTMs that combine the forget and input gates into a single update gate. This makes GRUs:

- Faster and more efficient than LSTMs
- Perform well on shorter sequences
- Require fewer computational resources

**Model Comparison (LSTM vs. GRU)**

To evaluate the differences between LSTM and GRU models, compare them using:

- Model Size & Number of Parameters : Use model.summary() to check the number of trainable parameters in each model.
- Training Speed & Efficiency : Measure training time per epoch for both models.

In [15]:
def build_gru_model(vocab_size, embedding_dim, rnn_units, batch_size):
    model = tf.keras.Sequential([
        tf.keras.Input(shape=(None,)),
        tf.keras.layers.Embedding(vocab_size, embedding_dim),
        tf.keras.layers.GRU(rnn_units, return_sequences=True, stateful=False),
        tf.keras.layers.Dense(vocab_size)
    ])
    return model

# Bidirectional Model


<img src="https://www.researchgate.net/publication/342646275/figure/fig4/AS:962238546464790@1606426955772/Comparison-between-LSTM-and-Bi-LSTM-networks-recreated-after-33.png" alt="Drawing"/>


**What is Bidirectional RNN?**

A Bidirectional Recurrent Neural Network (BiRNN) is an extension of a standard RNN that processes input sequences in both forward and backward directions. This means that at each time step, the model considers both past (left-to-right) and future (right-to-left) context, leading to better performance in many sequence-related tasks, including text generation.

**Why Use Bidirectional RNNs?**

Bidirectional RNNs are used because they capture dependencies in the input data from both directions, which is crucial for understanding context. This makes them particularly useful in tasks like language processing, where the meaning of a word can depend on both the words that come before and after it.

For example, in the sentence "The cat sat on the mat," the word "mat" is influenced by both the preceding words ("The cat sat on the") and what could potentially follow (e.g., "and looked at the mouse").

Example

Let's consider a sentence: "He opened the door."

Simple RNN: It processes the sentence from left to right, word by word:

He → opened → the → door
Each word is processed based on the previous word.

Bidirectional RNN: It processes the sentence in both directions:

Forward pass: He → opened → the → door

Backward pass: door → the → opened → He

The output at each time step is a combination of the information from both the forward and backward passes, allowing the network to understand the context better. For instance, knowing that "door" follows "the" helps confirm that "opened" likely refers to a physical action, not a metaphorical one.

**Choosing the Base RNN for Bidirectional Processing:**

One key flexibility of Bidirectional RNNs is that you can choose any recurrent architecture (LSTM, GRU, Simple RNN) as the base model.

- Bidirectional LSTM: Best for handling long-range dependencies.
- Bidirectional GRU: More computationally efficient than LSTM.
- Bidirectional Simple RNN: Less commonly used due to vanishing gradient issues.

Note: To determine the best base model for the Bidirectional RNN, you must evaluate the results from your previous experiments with different architectures (eg. LSTM and GRU).

Review previous results from training LSTM and GRU models.
Compare their performance in terms of:

- Training time per epoch
- Model size (number of parameters)
- Loss and accuracy on validation data
- Quality of generated text

Based on your findings, select the best-performing model as the base architecture for the Bidirectional RNN.


In [16]:
def build_birnn_model(vocab_size, embedding_dim, rnn_units, batch_size, rnn_type="LSTM"):
    if rnn_type.upper() == "LSTM":
        rnn_layer = tf.keras.layers.LSTM(rnn_units, return_sequences=True, stateful=False)
    elif rnn_type.upper() == "GRU":
        rnn_layer = tf.keras.layers.GRU(rnn_units, return_sequences=True, stateful=False)
    else:
        raise ValueError("rnn_type must be either 'LSTM' or 'GRU'")

    model = tf.keras.Sequential([
        tf.keras.Input(shape=(None,)),
        tf.keras.layers.Embedding(vocab_size, embedding_dim),
        tf.keras.layers.Bidirectional(rnn_layer),
        tf.keras.layers.Dense(vocab_size)
    ])
    return model

# Evaluating the Models

Model Selection

In [17]:
# Choose model type
model_type = "LSTM"  # Change to "GRU" or "BiLSTM"

if model_type == "LSTM":
    model = build_lstm_model(VOCAB_SIZE, EMBEDDING_DIM, RNN_UNITS, BATCH_SIZE)
elif model_type == "GRU":
    model = build_gru_model(VOCAB_SIZE, EMBEDDING_DIM, RNN_UNITS, BATCH_SIZE)
elif model_type == "BiLSTM":
    model = build_birnn_model(VOCAB_SIZE, EMBEDDING_DIM, RNN_UNITS, BATCH_SIZE)

print(f"Using {model_type} model for training.")

Using LSTM model for training.


In [18]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, None, 256)      │        16,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, None, 512)      │     1,574,912 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, None, 66)       │        33,858 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,625,666 (6.20 MB)

 Trainable params: 1,625,666 (6.20 MB)

 Non-trainable params: 0 (0.00 B)

Compile & Train the Model

In [19]:
def compile_and_train(model, path, EPOCHS = 30):
  def loss(labels, logits):
    return tf.keras.losses.sparse_categorical_crossentropy(labels, logits, from_logits=True)

  model.compile(optimizer='adam', loss=loss)

  checkpoint_dir = f'./{path}'
  os.makedirs(checkpoint_dir, exist_ok=True)

  checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt_{epoch}")

  checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_prefix + '.weights.h5',
    save_weights_only=True
  )

  history = model.fit(
    training_dataset,
    validation_data=validation_dataset,
    epochs=EPOCHS,
    callbacks=[checkpoint_callback]
)

  return history

In [20]:
class EpochTimeHistory(tf.keras.callbacks.Callback):
    def on_train_begin(self, logs=None):
        self.epoch_times = []

    def on_epoch_begin(self, epoch, logs=None):
        self.epoch_start_time = time.time()

    def on_epoch_end(self, epoch, logs=None):
        self.epoch_times.append(time.time() - self.epoch_start_time)

In [21]:
def compile_and_train_with_timing(model, path, EPOCHS=10):
    def loss(labels, logits):
        return tf.keras.losses.sparse_categorical_crossentropy(labels, logits, from_logits=True)

    model.compile(optimizer='adam', loss=loss)

    checkpoint_dir = f'./{path}'
    os.makedirs(checkpoint_dir, exist_ok=True)

    checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt_{epoch}")

    checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
        filepath=checkpoint_prefix + '.weights.h5',
        save_weights_only=True
    )

    time_callback = EpochTimeHistory()

    history = model.fit(
        training_dataset,
        validation_data=validation_dataset,
        epochs=EPOCHS,
        callbacks=[checkpoint_callback, time_callback]
    )

    return history, time_callback.epoch_times

# Generate Text Using the Model

In [22]:
class OneStepTextGenerator(tf.keras.Model):
    def __init__(self, model, chars_from_ids, ids_from_chars, temperature=1.0):
        super().__init__()
        self.model = model
        self.chars_from_ids = chars_from_ids
        self.ids_from_chars = ids_from_chars
        self.temperature = temperature

    def generate_text(self, start_string, num_generate=1000):
        input_eval = tf.expand_dims(self.ids_from_chars(tf.strings.unicode_split(start_string, 'UTF-8')), 0)
        text_generated = []

        states = None

        for _ in range(num_generate):
            predictions = self.model(input_eval)
            predictions = predictions[:, -1, :] / self.temperature
            predicted_id = tf.random.categorical(predictions, num_samples=1)[-1, 0].numpy()
            input_eval = tf.expand_dims([predicted_id], 0)
            text_generated.append(self.chars_from_ids(predicted_id).numpy().decode('utf-8'))

        return start_string + ''.join(text_generated)

# Evaluate Model Performance

**1- Perplexity (PP)**

TODO: Measures uncertainty in predicting next character for each model and leave the results in the Notebook.

***Lower values = better model***

In [23]:
def perplexity(logits, labels):
    loss = tf.keras.losses.sparse_categorical_crossentropy(labels, logits, from_logits=True)
    return np.exp(np.mean(loss))

def evaluate_model_perplexity(model, dataset):
    total_loss = 0.0
    total_tokens = 0

    for input_batch, label_batch in dataset:
        logits = model(input_batch, training=False)
        loss = perplexity(logits, label_batch)
        total_loss += np.log(loss)
        total_tokens += tf.size(label_batch).numpy()

    avg_loss = total_loss / total_tokens
    return np.exp(avg_loss)

**2- Text Coherence & Fluency**

Subjective evaluation by examining generated text

**TODO: Generate at least 3 Samples (one from each model) and leave them in the Notebook.**

**3- BLEU Score**

TODO: Measures similarity between generated and real text for each model and leave the results in the Notebook.

In [24]:
def compute_bleu(reference, generated):
    return sentence_bleu([list(reference)], list(generated))

def evaluate_model_bleu(model, dataset):
    bleu_scores = []

    for input_batch, label_batch in dataset:
        logits = model(input_batch, training=False)
        predictions = tf.argmax(logits, axis=-1)

        label_batch = label_batch.numpy()

        for pred_seq, true_seq in zip(predictions, label_batch):
            # Decode sequences to text
            pred_text = text_from_ids(pred_seq, chars_from_ids).numpy().decode('utf-8')
            true_text = text_from_ids(true_seq, chars_from_ids).numpy().decode('utf-8')

            # Compute BLEU score for each sample
            bleu = compute_bleu(true_text, pred_text)
            bleu_scores.append(bleu)

    return np.mean(bleu_scores)

In [25]:
lstm_trials = [
    {"name": "LSTM_trial_1", "embedding_dim": 128, "rnn_units": 256, "num_layers": 1},
    {"name": "LSTM_trial_2", "embedding_dim": 256, "rnn_units": 512, "num_layers": 1},
    {"name": "LSTM_trial_3", "embedding_dim": 256, "rnn_units": 512, "num_layers": 2}
]

lstm_trial_results = []

In [26]:
for trial in lstm_trials:
    print(f"Running {trial['name']}")

    if trial["num_layers"] == 1:
        model = build_lstm_model(VOCAB_SIZE, trial["embedding_dim"], trial["rnn_units"], BATCH_SIZE)
    else:
        model = build_stacked_lstm_model(VOCAB_SIZE, trial["embedding_dim"], trial["rnn_units"], BATCH_SIZE, num_layers=trial["num_layers"])

    num_params = model.count_params()

    history, epoch_times = compile_and_train_with_timing(model, trial["name"].lower(), EPOCHS=10)

    perplexity_score = evaluate_model_perplexity(model, validation_dataset)
    bleu_score = evaluate_model_bleu(model, validation_dataset)

    lstm_trial_results.append({
        "name": trial["name"],
        "embedding_dim": trial["embedding_dim"],
        "rnn_units": trial["rnn_units"],
        "num_layers": trial["num_layers"],
        "params": num_params,
        "avg_epoch_time": np.mean(epoch_times),
        "final_train_loss": history.history["loss"][-1],
        "final_val_loss": history.history["val_loss"][-1],
        "perplexity": perplexity_score,
        "bleu": bleu_score
    })

    print("Parameters:", num_params)
    print("Average epoch time:", np.mean(epoch_times))
    print("Final train loss:", history.history["loss"][-1])
    print("Final val loss:", history.history["val_loss"][-1])
    print("Perplexity:", perplexity_score)
    print("BLEU:", bleu_score)
    print("=" * 80)

Running LSTM_trial_1
Epoch 1/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 14s 31ms/step - loss: 2.7481 - val_loss: 2.2770
Epoch 2/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 7s 31ms/step - loss: 2.1399 - val_loss: 2.0240
Epoch 3/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 6s 30ms/step - loss: 1.9477 - val_loss: 1.8797
Epoch 4/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 10s 29ms/step - loss: 1.8304 - val_loss: 1.7802
Epoch 5/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 7s 32ms/step - loss: 1.7470 - val_loss: 1.7101
Epoch 6/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 6s 30ms/step - loss: 1.6862 - val_loss: 1.6576
Epoch 7/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 7s 31ms/step - loss: 1.6397 - val_loss: 1.6156
Epoch 8/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 11s 34ms/step - loss: 1.6015 - val_loss: 1.5803
Epoch 9/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 6s 30ms/step - loss: 1.5689 - val_loss: 1.5497
Epoch 10/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 7s 33ms/step - loss: 1.5423 - val_loss: 1.5258


/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)


Parameters: 419650
Average epoch time: 8.09930591583252
Final train loss: 1.542301058769226
Final val loss: 1.5257656574249268
Perplexity: 1.0002384432643905
BLEU: 0.2673813682377279
Running LSTM_trial_2
Epoch 1/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 12s 57ms/step - loss: 2.6300 - val_loss: 2.1578
Epoch 2/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 10s 54ms/step - loss: 1.9956 - val_loss: 1.8577
Epoch 3/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 10s 53ms/step - loss: 1.7752 - val_loss: 1.6959
Epoch 4/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 11s 54ms/step - loss: 1.6490 - val_loss: 1.5948
Epoch 5/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 10s 54ms/step - loss: 1.5662 - val_loss: 1.5262
Epoch 6/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 10s 54ms/step - loss: 1.5084 - val_loss: 1.4762
Epoch 7/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 11s 56ms/step - loss: 1.4647 - val_loss: 1.4379
Epoch 8/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 10s 52ms/step - loss: 1.4295 - val_loss: 1.4062
Epoch 9/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 10s 52ms/step - loss: 1.4014 - val_loss: 1.3794
Epoc

In [27]:
for result in lstm_trial_results:
    print(result)

{'name': 'LSTM_trial_1', 'embedding_dim': 128, 'rnn_units': 256, 'num_layers': 1, 'params': 419650, 'avg_epoch_time': np.float64(8.09930591583252), 'final_train_loss': 1.542301058769226, 'final_val_loss': 1.5257656574249268, 'perplexity': np.float64(1.0002384432643905), 'bleu': np.float64(0.2673813682377279)}
{'name': 'LSTM_trial_2', 'embedding_dim': 256, 'rnn_units': 512, 'num_layers': 1, 'params': 1625666, 'avg_epoch_time': np.float64(10.346089196205138), 'final_train_loss': 1.3766707181930542, 'final_val_loss': 1.3524525165557861, 'perplexity': np.float64(1.0002111755863123), 'bleu': np.float64(0.31651905780042533)}
{'name': 'LSTM_trial_3', 'embedding_dim': 256, 'rnn_units': 512, 'num_layers': 2, 'params': 3724866, 'avg_epoch_time': np.float64(17.253272366523742), 'final_train_loss': 1.307391881942749, 'final_val_loss': 1.2754327058792114, 'perplexity': np.float64(1.000199359828459), 'bleu': np.float64(0.33506971126956014)}


In [28]:
best_lstm_result = min(lstm_trial_results, key=lambda x: x["perplexity"])
best_lstm_result

{'name': 'LSTM_trial_3',
 'embedding_dim': 256,
 'rnn_units': 512,
 'num_layers': 2,
 'params': 3724866,
 'avg_epoch_time': np.float64(17.253272366523742),
 'final_train_loss': 1.307391881942749,
 'final_val_loss': 1.2754327058792114,
 'perplexity': np.float64(1.000199359828459),
 'bleu': np.float64(0.33506971126956014)}

In [29]:
if best_lstm_result["num_layers"] == 1:
    best_lstm_model = build_lstm_model(
        VOCAB_SIZE,
        best_lstm_result["embedding_dim"],
        best_lstm_result["rnn_units"],
        BATCH_SIZE
    )
else:
    best_lstm_model = build_stacked_lstm_model(
        VOCAB_SIZE,
        best_lstm_result["embedding_dim"],
        best_lstm_result["rnn_units"],
        BATCH_SIZE,
        num_layers=best_lstm_result["num_layers"]
    )

best_lstm_model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ (None, None, 256)      │        16,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ (None, None, 512)      │     1,574,912 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_6 (LSTM)                   │ (None, None, 512)      │     2,099,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, None, 66)       │        33,858 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,724,866 (14.21 MB)

 Trainable params: 3,724,866 (14.21 MB)

 Non-trainable params: 0 (0.00 B)

In [30]:
best_lstm_history, best_lstm_epoch_times = compile_and_train_with_timing(best_lstm_model, "best_lstm_model", EPOCHS=10)
best_lstm_perplexity = evaluate_model_perplexity(best_lstm_model, validation_dataset)
best_lstm_bleu = evaluate_model_bleu(best_lstm_model, validation_dataset)

print("Best LSTM Perplexity:", best_lstm_perplexity)
print("Best LSTM BLEU:", best_lstm_bleu)
print("Best LSTM Avg Epoch Time:", np.mean(best_lstm_epoch_times))

Epoch 1/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 22s 97ms/step - loss: 2.8141 - val_loss: 2.2426
Epoch 2/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 17s 98ms/step - loss: 2.0486 - val_loss: 1.8819
Epoch 3/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 18s 100ms/step - loss: 1.7818 - val_loss: 1.6810
Epoch 4/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 20s 96ms/step - loss: 1.6275 - val_loss: 1.5643
Epoch 5/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 17s 99ms/step - loss: 1.5285 - val_loss: 1.4784
Epoch 6/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 17s 97ms/step - loss: 1.4595 - val_loss: 1.4210
Epoch 7/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 17s 97ms/step - loss: 1.4117 - val_loss: 1.3792
Epoch 8/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 17s 98ms/step - loss: 1.3719 - val_loss: 1.3389
Epoch 9/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 17s 96ms/step - loss: 1.3399 - val_loss: 1.3076
Epoch 10/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 17s 97ms/step - loss: 1.3111 - val_loss: 1.2789
Best LSTM Perplexity: 1.0001999067006313
Best LSTM BLEU: 0.3367749357443902
Best LSTM Avg Epoch Time: 17.7635311365127

In [31]:
gru_model = build_gru_model(VOCAB_SIZE, best_lstm_result["embedding_dim"], best_lstm_result["rnn_units"], BATCH_SIZE)
gru_model.summary()

gru_history, gru_epoch_times = compile_and_train_with_timing(gru_model, "gru_model", EPOCHS=10)
gru_perplexity = evaluate_model_perplexity(gru_model, validation_dataset)
gru_bleu = evaluate_model_bleu(gru_model, validation_dataset)

print("GRU Perplexity:", gru_perplexity)
print("GRU BLEU:", gru_bleu)
print("GRU Avg Epoch Time:", np.mean(gru_epoch_times))

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ (None, None, 256)      │        16,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, None, 512)      │     1,182,720 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, None, 66)       │        33,858 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,233,474 (4.71 MB)

 Trainable params: 1,233,474 (4.71 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 14s 50ms/step - loss: 2.4123 - val_loss: 2.0059
Epoch 2/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 9s 47ms/step - loss: 1.8475 - val_loss: 1.7103
Epoch 3/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 9s 50ms/step - loss: 1.6361 - val_loss: 1.5566
Epoch 4/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 9s 47ms/step - loss: 1.5231 - val_loss: 1.4749
Epoch 5/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 10s 47ms/step - loss: 1.4524 - val_loss: 1.4121
Epoch 6/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 10s 47ms/step - loss: 1.4053 - val_loss: 1.3716
Epoch 7/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 9s 50ms/step - loss: 1.3675 - val_loss: 1.3345
Epoch 8/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 9s 47ms/step - loss: 1.3380 - val_loss: 1.3078
Epoch 9/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 10s 47ms/step - loss: 1.3145 - val_loss: 1.2863
Epoch 10/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 9s 47ms/step - loss: 1.2923 - val_loss: 1.2635
GRU Perplexity: 1.00019775159892
GRU BLEU: 0.3383060355982048
GRU Avg Epoch Time: 9.822317910194396


In [32]:
print("LSTM Parameters:", best_lstm_model.count_params())
print("GRU Parameters:", gru_model.count_params())
print("LSTM Avg Epoch Time:", np.mean(best_lstm_epoch_times))
print("GRU Avg Epoch Time:", np.mean(gru_epoch_times))
print("LSTM Final Val Loss:", best_lstm_history.history["val_loss"][-1])
print("GRU Final Val Loss:", gru_history.history["val_loss"][-1])
print("LSTM Perplexity:", best_lstm_perplexity)
print("GRU Perplexity:", gru_perplexity)
print("LSTM BLEU:", best_lstm_bleu)
print("GRU BLEU:", gru_bleu)

LSTM Parameters: 3724866
GRU Parameters: 1233474
LSTM Avg Epoch Time: 17.763531136512757
GRU Avg Epoch Time: 9.822317910194396
LSTM Final Val Loss: 1.2789382934570312
GRU Final Val Loss: 1.2635133266448975
LSTM Perplexity: 1.0001999067006313
GRU Perplexity: 1.00019775159892
LSTM BLEU: 0.3367749357443902
GRU BLEU: 0.3383060355982048


In [33]:
best_base_rnn = "LSTM" if best_lstm_perplexity <= gru_perplexity else "GRU"
print("Best base model for Bidirectional RNN:", best_base_rnn)

Best base model for Bidirectional RNN: GRU


In [34]:
birnn_model = build_birnn_model(
    VOCAB_SIZE,
    best_lstm_result["embedding_dim"],
    best_lstm_result["rnn_units"],
    BATCH_SIZE,
    rnn_type=best_base_rnn
)

birnn_model.summary()

birnn_history, birnn_epoch_times = compile_and_train_with_timing(birnn_model, "birnn_model", EPOCHS=10)
birnn_perplexity = evaluate_model_perplexity(birnn_model, validation_dataset)
birnn_bleu = evaluate_model_bleu(birnn_model, validation_dataset)

print("BiRNN Perplexity:", birnn_perplexity)
print("BiRNN BLEU:", birnn_bleu)
print("BiRNN Avg Epoch Time:", np.mean(birnn_epoch_times))

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_6 (Embedding)         │ (None, None, 256)      │        16,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, None, 1024)     │     2,365,440 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, None, 66)       │        67,650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,449,986 (9.35 MB)

 Trainable params: 2,449,986 (9.35 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 16s 83ms/step - loss: 0.7066 - val_loss: 0.0316
Epoch 2/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 21s 83ms/step - loss: 0.0273 - val_loss: 0.0248
Epoch 3/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 20s 81ms/step - loss: 0.0241 - val_loss: 0.0230
Epoch 4/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 15s 80ms/step - loss: 0.0228 - val_loss: 0.0218
Epoch 5/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 15s 82ms/step - loss: 0.0218 - val_loss: 0.0208
Epoch 6/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 15s 83ms/step - loss: 0.0209 - val_loss: 0.0198
Epoch 7/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 14s 80ms/step - loss: 0.0202 - val_loss: 0.0190
Epoch 8/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 21s 83ms/step - loss: 0.0193 - val_loss: 0.0180
Epoch 9/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 14s 82ms/step - loss: 0.0185 - val_loss: 0.0167
Epoch 10/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 15s 85ms/step - loss: 0.0173 - val_loss: 0.0155
BiRNN Perplexity: 1.0000024326840984
BiRNN BLEU: 0.995179338438796
BiRNN Avg Epoch Time: 16.510631489753724


In [35]:
print("Best LSTM Parameters:", best_lstm_model.count_params())
print("GRU Parameters:", gru_model.count_params())
print("BiRNN Parameters:", birnn_model.count_params())

print("Best LSTM Avg Epoch Time:", np.mean(best_lstm_epoch_times))
print("GRU Avg Epoch Time:", np.mean(gru_epoch_times))
print("BiRNN Avg Epoch Time:", np.mean(birnn_epoch_times))

print("Best LSTM Final Val Loss:", best_lstm_history.history["val_loss"][-1])
print("GRU Final Val Loss:", gru_history.history["val_loss"][-1])
print("BiRNN Final Val Loss:", birnn_history.history["val_loss"][-1])

print("Best LSTM Perplexity:", best_lstm_perplexity)
print("GRU Perplexity:", gru_perplexity)
print("BiRNN Perplexity:", birnn_perplexity)

print("Best LSTM BLEU:", best_lstm_bleu)
print("GRU BLEU:", gru_bleu)
print("BiRNN BLEU:", birnn_bleu)

Best LSTM Parameters: 3724866
GRU Parameters: 1233474
BiRNN Parameters: 2449986
Best LSTM Avg Epoch Time: 17.763531136512757
GRU Avg Epoch Time: 9.822317910194396
BiRNN Avg Epoch Time: 16.510631489753724
Best LSTM Final Val Loss: 1.2789382934570312
GRU Final Val Loss: 1.2635133266448975
BiRNN Final Val Loss: 0.015516944229602814
Best LSTM Perplexity: 1.0001999067006313
GRU Perplexity: 1.00019775159892
BiRNN Perplexity: 1.0000024326840984
Best LSTM BLEU: 0.3367749357443902
GRU BLEU: 0.3383060355982048
BiRNN BLEU: 0.995179338438796


In [36]:
lstm_generator = OneStepTextGenerator(best_lstm_model, chars_from_ids, ids_from_chars, temperature=0.8)
gru_generator = OneStepTextGenerator(gru_model, chars_from_ids, ids_from_chars, temperature=0.8)
birnn_generator = OneStepTextGenerator(birnn_model, chars_from_ids, ids_from_chars, temperature=0.8)

print("LSTM Sample:\n")
print(lstm_generator.generate_text(start_string="ROMEO: ", num_generate=300))

print("\n" + "=" * 100 + "\n")

print("GRU Sample:\n")
print(gru_generator.generate_text(start_string="ROMEO: ", num_generate=300))

print("\n" + "=" * 100 + "\n")

print("BiRNN Sample:\n")
print(birnn_generator.generate_text(start_string="ROMEO: ", num_generate=300))

LSTM Sample:

ROMEO: I tll n,


CI ave besth t anoous tus y ce t
WOMe d mat se war bome s, bemsh t onthe'llth chesousthe moncknd anoper'd thind t the tllderamar VI
Th l athe s henbuckirin d, pe y thes n non th tlame g w tloful allend d gr misuacan.
An te wieange yorele pr'senof thinous stere ancere bl fers mecout, ord i


GRU Sample:

ROMEO: HERO&Payo; l blle soubeloutachin sino thowe f f t thant.
CA.
Y pyofurishand in thiste nd blis, I wing myoffoupro?
K:
O:
ous te tho the monof atin tanghtongodoris d th h andy s d y bettoweree anghengl whe ber,
The swochis, pe, mes whiba, se omakie s thatwearin pad ase sthe s futhe s aled me:
Wesisthe


BiRNN Sample:

ROMEO: I
Hofy str:
S
S
Y V&3
Mand pre:
Vest y m:
Hele n:
CKCy'V aked H:
Pupouud,
ME
R:
Yes,
Hecke y inety,
Cboos,
Wh Pe magnththe,
He:
ICheme y an.
ANG s alache,
Butinged He I'$LANG;
S
Rman,
Bk Bus'ligowfigurdid ad hompt:
JBugan, halar qJGwh,
ESS's blis, anthy s achichinchhe O
D may mergharowoncch BENonero


In [37]:
print("LSTM Sample 2:\n")
print(lstm_generator.generate_text(start_string="JULIET: ", num_generate=300))

print("\n" + "=" * 100 + "\n")

print("GRU Sample 2:\n")
print(gru_generator.generate_text(start_string="KING: ", num_generate=300))

print("\n" + "=" * 100 + "\n")

print("BiRNN Sample 2:\n")
print(birnn_generator.generate_text(start_string="QUEEN: ", num_generate=300))

LSTM Sample 2:

JULIET: MECUSthy ' houle.
TOgr arsthe gastowiestomy br, wis couind ouich.

The lanin way athishat rest d, wit,

ARAroustous uerin.
BAn ssthor be fl anchanor, t d HAngudof f me thout tomang ce atthomigomerl hand IZhe burr'the oros IO:


IORERThishe.
Then, ie ak.
Tis sth abu ped.
The my chert demano theadscis


GRU Sample 2:

KING: beane, hest t courayourofalange thif;
Wis whes, izerellerese hevente be t;
LAnoupo me bound beaner,

THAPEdomof be d sengow ofe;
INENInge torenthend fowe herizeame,
INAn IO, tanes merigo sty sion batou jusel mowiladbe we ooushed,
The hed bed thee cayo;
LAND then t h thoupo l t t mepoulllend t, setin


BiRNN Sample 2:

QUEEN: ICt arinth me ply s wine y,
My am,
J,
SThis
NG t sid nck m,
Yf,
Whth me,
MONd D,
PCame:
O
Sond;
My ING LANG wy,
Tuprantin te ty:
M bd Ck:
Hine,
Pu win:
MFfan:
M,
My,
CK:
EX.
Whigod wy m,
MJLbendsG s nge G'schity Gwn, Whed
PFagh merilutinon'TwXd Xfontho Y
My Malode by m hew thit y grnwe by y Hy y LIO


In [38]:
if birnn_perplexity < min(best_lstm_perplexity, gru_perplexity):
    best_overall_model = "BiRNN"
elif best_lstm_perplexity <= gru_perplexity:
    best_overall_model = "LSTM"
else:
    best_overall_model = "GRU"

print("Best overall model based on perplexity:", best_overall_model)
print("Bidirectional impact: compare BiRNN results against the best unidirectional model using validation loss, perplexity, BLEU, and generated text quality.")

Best overall model based on perplexity: BiRNN
Bidirectional impact: compare BiRNN results against the best unidirectional model using validation loss, perplexity, BLEU, and generated text quality.
